<a href="https://colab.research.google.com/github/Jimber120/DA-Project-Member-3/blob/main/work/notebooks/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Grouped Split, Baseline Benchmarking & Model Training

**Project:** FlyRank Ranking Signal Analysis — Predicting Continuous Search Visibility  
**Pipeline Stage:** Notebook 02 of 03 (Model Training & Benchmark Optimization)  
**Target Variable:** Continuous Search Visibility $\log(1 + \text{impressions\_90d})$ on Held-Out Enterprise Client Portfolios  
**Standardized Split:** Grouped Holdout (`GroupShuffleSplit`, `test_size=0.20`, `random_state=42`) with Zero Client Domain Overlap  

---

### Notebook Architecture & Objectives
This notebook is your starting template for model training and optimization:
1. **Grouped Holdout Split Design:** Enforce a strict zero-client-leakage partition on `client_id` with programmatic disjointness assertions.
2. **Transparent Heuristic Baselines:** Fit Baseline 1 (Mean Dummy Regressor) and Baseline 2 (Univariate Word Count Rule) to establish the floor to beat.
3. **High-Accuracy Model Training:**
   - Fit candidate algorithms (e.g., cross-validated Polynomial Interaction Ridge, HistGradientBoosting Regressor, or Stacking Ensemble).
   - Maximize generalization accuracy on unseen client domains.
4. **Payload Export:** Export model predictions, test set ground truth, and metadata to `work/outputs/model_evaluation_payload.pkl` for evaluation in Notebook 03.

## 1. Grouped Holdout Split Design (Zero Client Leakage)

### 1.1 The Multi-Tenant Generalization Challenge
In multi-tenant SaaS platforms, standard random row splits produce severe **client leakage**: content items from the same client share identical domain authority, root backlink velocity, CMS infrastructure, and brand equity. A random split allows models to memorize client-level baseline volume, inflating test performance while collapsing on brand-new client domains.

### 1.2 The Client Holdout Standard
To guarantee 100% scientific validity:
* We partition strictly by `client_id` using `GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)`.
* We assert that training clients and test clients are mutually exclusive:
  $$\text{set}(\text{clients}_{\text{train}}) \cap \text{set}(\text{clients}_{\text{test}}) = \emptyset$$
* Test metrics reflect true generalization performance on entirely unseen enterprise domains.

In [19]:
# Code Cell 1: Load Features and Execute Grouped Split (reads from local Colab session storage)
import os
import pickle
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:                                   # scikit-learn < 1.4
    def root_mean_squared_error(y_true, y_pred):
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

OUT_DIR = os.path.join("work", "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
parquet_path = os.path.join(OUT_DIR, "prepared_features.parquet")
csv_path = os.path.join(OUT_DIR, "prepared_features.csv")

df = None
if os.path.exists(parquet_path):
    try:
        df = pd.read_parquet(parquet_path)
    except Exception:
        df = None
if df is None and os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
if df is None:
    raise FileNotFoundError("prepared_features not found in work/outputs/. Run Notebook 01 first "
                            "in this SAME Colab session (the runtime filesystem is temporary).")

# Metadata columns are never features
META_COLS = [c for c in df.columns if c in ("content_id", "client_id", "target_log_impressions") or c.startswith("meta_")]
feature_cols = [c for c in df.columns if c not in META_COLS]

X = df[feature_cols]
y = df["target_log_impressions"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)
clients_train = groups.iloc[train_idx].reset_index(drop=True)
clients_test = groups.iloc[test_idx].reset_index(drop=True)
meta_test = df.iloc[test_idx][META_COLS].reset_index(drop=True)

assert set(clients_train).isdisjoint(set(clients_test)), "Client leakage detected between train and test sets!"
assert not any(c in X_train.columns for c in ("content_id", "client_id", "target_log_impressions")), "Identifier/target in features!"

print(f"Train Cohort: {len(X_train):,} rows across {clients_train.nunique()} clients")
print(f"Test Cohort:  {len(X_test):,} rows across {clients_test.nunique()} unseen clients")
print(f"Feature count: {X_train.shape[1]}")
print("\nTarget distribution:")
print(pd.DataFrame({"train": y_train.describe(), "test": y_test.describe()}).round(3))

Train Cohort: 23,837 rows across 25 clients
Test Cohort:  6,163 rows across 7 unseen clients
Feature count: 54

Target distribution:
            train       test
count 23,837.0000 6,163.0000
mean       6.3090     5.7240
std        2.6770     2.6830
min        0.6930     0.6930
25%        4.7000     3.4340
50%        6.7030     6.0640
75%        8.2900     7.8080
max       13.1560    13.1570


## 2. Transparent Heuristic Baseline Benchmarks

To verify that machine learning provides genuine predictive lift rather than fitting noise, evaluate two non-ML baselines on the exact same test cohort:
1. **Baseline 1 (Mean Dummy Regressor):** Always predicts the training set target mean $\bar{y}_{\text{train}}$. (Yields $R^2 = 0.0$ on train; negative $R^2$ on held-out clients with a lower mean).
2. **Baseline 2 (Univariate Word Count Rule):** A simple single-variable OLS regression on `log_word_count`, reflecting the common SEO belief that "longer articles rank higher and gain more visibility".

In [20]:
# Code Cell 2: Fit and Evaluate Standardized Heuristic Baselines
from sklearn.linear_model import LinearRegression


def evaluate_predictions(y_true, y_pred, model_name):
    r2 = r2_score(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    # A constant prediction has no rank information: define rho = 0
    rho = 0.0 if np.ptp(np.asarray(y_pred)) == 0 else spearmanr(y_true, y_pred)[0]
    return {"Model": model_name, "R2": r2, "RMSE": rmse, "MAE": mae, "Spearman_Rho": rho}


# Baseline 1: always predict the training mean
y_pred_b1 = np.full(len(y_test), y_train.mean())

# Baseline 2: univariate OLS on log_word_count (the "longer is better" editorial rule)
assert "log_word_count" in X_train.columns, "log_word_count missing from the feature matrix"
b2 = LinearRegression().fit(X_train[["log_word_count"]], y_train)
y_pred_b2 = b2.predict(X_test[["log_word_count"]])

baseline_results = pd.DataFrame([
    evaluate_predictions(y_test, y_pred_b1, "Baseline 1 (Mean Dummy)"),
    evaluate_predictions(y_test, y_pred_b2, "Baseline 2 (Word Count Rule)"),
])
print("=== Baseline performance on held-out client domains ===")
display(baseline_results.round(4))
print("\nNote: a slightly negative R2 for Baseline 1 is expected - unseen clients have a different average volume "
      "than the training clients. It is not a bug; it is the performance floor.")

=== Baseline performance on held-out client domains ===


,Model,R2,RMSE,MAE,Spearman_Rho
0,Baseline 1 (Mean Dummy),-0.0476,2.7457,2.2757,0.0000
1,Baseline 2 (Word Count Rule),-0.0112,2.6976,2.2441,0.0429



Note: a slightly negative R2 for Baseline 1 is expected - unseen clients have a different average volume than the training clients. It is not a bug; it is the performance floor.


## 3. High-Accuracy Model Training

### 3.1 Model Architecture Strategy
To achieve the highest honest accuracy without data leakage, consider:
1. **Regularized Linear Models with Interactions:** Pairwise interaction terms ($x_i \times x_j$) across primary signals regularized with `RidgeCV`.
2. **Gradient Boosted Decision Trees (`HistGradientBoostingRegressor`):** Captures sharp non-linear thresholds, step-function jumps, and missingness natively.
3. **Champion Blended Ensemble:** Blending linear macro-trends with localized tree splits ($w \cdot \hat{y}_{\text{Ridge}} + (1-w) \cdot \hat{y}_{\text{HGBR}}$) to maximize variance explained.

In [21]:
# Code Cell 3: Machine Learning Model Training & Optimization
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor

# ---------------- Candidate 1: Ridge with pairwise interactions among primary signals ----------------
PRIMARY = [c for c in ["serp_rank_power", "log_search_volume", "log_word_count",
                       "log_internal_links_in", "log_external_backlinks"] if c in X_train.columns]
OTHER = [c for c in X_train.columns if c not in PRIMARY]
print(f"Primary interaction signals: {PRIMARY}")

poly_block = Pipeline([
    ("scale", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
])
preprocess = ColumnTransformer([
    ("poly", poly_block, PRIMARY),
    ("rest", StandardScaler(), OTHER),       # every other feature enters linearly
])
# Cross-validate alpha by CLIENT so the regularization is also tuned for unseen domains
cv_splits = list(GroupKFold(n_splits=5).split(X_train, y_train, groups=clients_train))
ridge_model = Pipeline([
    ("prep", preprocess),
    ("ridge", RidgeCV(alphas=np.logspace(-2, 3, 20), cv=cv_splits)),
])
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)
print(f"RidgeCV selected alpha = {ridge_model.named_steps['ridge'].alpha_:.4f}")

# ---------------- Candidate 2: HistGradientBoostingRegressor ----------------
hgbr = HistGradientBoostingRegressor(
    max_iter=250,
    learning_rate=0.06,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.5,
    random_state=42,
)
hgbr.fit(X_train, y_train)
y_pred_hgbr = hgbr.predict(X_test)

# ---------------- Candidate 3: Champion convex blend (weights fixed a priori, NOT tuned on test) ----------------
W_RIDGE, W_HGBR = 0.50, 0.50
y_pred_ensemble = W_RIDGE * y_pred_ridge + W_HGBR * y_pred_hgbr

model_results = pd.DataFrame([
    evaluate_predictions(y_test, y_pred_b1, "Baseline 1 (Mean Dummy)"),
    evaluate_predictions(y_test, y_pred_b2, "Baseline 2 (Word Count Rule)"),
    evaluate_predictions(y_test, y_pred_ridge, "Ridge Interactions"),
    evaluate_predictions(y_test, y_pred_hgbr, "HistGradientBoosting"),
    evaluate_predictions(y_test, y_pred_ensemble, f"Champion Ensemble ({W_RIDGE:.2f} Ridge + {W_HGBR:.2f} HGBR)"),
])
base_rmse = model_results.loc[0, "RMSE"]
model_results["RMSE_Reduction_%"] = (base_rmse - model_results["RMSE"]) / base_rmse * 100
print("\n=== Held-out client test metrics ===")
display(model_results.round(4))

Primary interaction signals: ['serp_rank_power', 'log_search_volume', 'log_word_count']
RidgeCV selected alpha = 1000.0000

=== Held-out client test metrics ===


,Model,R2,RMSE,MAE,Spearman_Rho,RMSE_Reduction_%
0,Baseline 1 (Mean Dummy),-0.0476,2.7457,2.2757,0.0000,0.0000
1,Baseline 2 (Word Count Rule),-0.0112,2.6976,2.2441,0.0429,1.7510
2,Ridge Interactions,0.3834,2.1065,1.7926,0.6068,23.2781
3,HistGradientBoosting,0.5315,1.8362,1.4772,0.7201,33.1251
4,Champion Ensemble (0.50 Ridge + 0.50 HGBR),0.4925,1.9111,1.6028,0.7019,30.3948


In [22]:
# Code Cell 4: Export Model Evaluation Payload (local Colab session storage)
payload = {
    "X_test": X_test,
    "y_test": y_test,
    "clients_test": clients_test,
    "meta_test": meta_test,                      # content_id, client_id and raw context columns for Notebook 03
    "preds": {
        "Baseline 1 (Mean Dummy)": y_pred_b1,
        "Baseline 2 (Word Count Rule)": y_pred_b2,
        "Ridge Interactions": y_pred_ridge,
        "HistGradientBoosting": y_pred_hgbr,
        "Champion Stacking Ensemble": y_pred_ensemble,
    },
    "info": {
        "feature_names": list(X_train.columns),
        "ensemble_weights": {"ridge": W_RIDGE, "hgbr": W_HGBR},
        "ridge_alpha": float(ridge_model.named_steps["ridge"].alpha_),
        "n_train": int(len(X_train)), "n_test": int(len(X_test)),
        "n_clients_train": int(clients_train.nunique()), "n_clients_test": int(clients_test.nunique()),
    },
}

payload_path = os.path.join(OUT_DIR, "model_evaluation_payload.pkl")
with open(payload_path, "wb") as f:
    pickle.dump(payload, f)

pred_df = meta_test[["content_id", "client_id"]].copy()
pred_df["y_true"] = y_test.values
for name, p in payload["preds"].items():
    pred_df[name] = p
pred_path = os.path.join(OUT_DIR, "model_test_predictions.csv")
pred_df.to_csv(pred_path, index=False)

print("Evaluation payload successfully exported.")
for p in (payload_path, pred_path):
    print(f"  {p}  ({os.path.getsize(p) / 1024:,.1f} KB)")

Evaluation payload successfully exported.
  work/outputs/model_evaluation_payload.pkl  (3,356.6 KB)
  work/outputs/model_test_predictions.csv  (887.5 KB)
